# Logistic Regression Baseline
Leakage-aware models, complete-case clinical inference, imputation sensitivity, 5-fold CV, PR-AUC baselines, and survey-weighted prevalence.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import StratifiedKFold,cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
import statsmodels.api as sm
DATA_DIR=Path('../../data'); RESULTS_DIR=Path('./results'); RESULTS_DIR.mkdir(exist_ok=True)
df=pd.read_csv(DATA_DIR/'ensanut_integrated.csv',low_memory=False)
def num(s): return pd.to_numeric(s.astype('string').str.replace(',','.',regex=False).str.strip(),errors='coerce')
for c in ['WEIGHT_FINAL','HEIGHT_FINAL','SEXO','EDAD','CINTURA21_1','CIRCPANTORRILLA19_1','CRP','FERRITINA','HCST','VIT_B12','VIT_D','ENT','REGION','FACTOR','FACTOR_EXPANSION','P8_9']:
    if c in df: df[c]=num(df[c])
df=df[df.EDAD.ge(20)].copy(); df=df.loc[~(df.SEXO.eq(2)&df.P8_9.eq(1))] if 'P8_9' in df else df
h=df.HEIGHT_FINAL/100; df['BMI']=df.WEIGHT_FINAL/h**2; male=df.SEXO.eq(1).astype(float)
df['ASM']=.244*df.WEIGHT_FINAL+7.8*h+6.6*male-.098*df.EDAD-3.3; df['ASMI']=df.ASM/h**2; df['BFP']=1.2*df.BMI+.23*df.EDAD-10.8*male-5.4
cc=(df.SEXO.eq(1)&df.CIRCPANTORRILLA19_1.lt(34))|(df.SEXO.eq(2)&df.CIRCPANTORRILLA19_1.lt(33)); df['SO_SIMP_1']=np.where(df.BMI.notna()&df.CIRCPANTORRILLA19_1.notna(),((df.BMI.ge(30))&cc).astype(float),np.nan)
high=(df.SEXO.eq(1)&df.BFP.gt(25))|(df.SEXO.eq(2)&df.BFP.gt(35)); low=(df.SEXO.eq(1)&df.ASMI.lt(7))|(df.SEXO.eq(2)&df.ASMI.lt(5.7)); df['SO_ESPEN']=np.where(df.BFP.notna()&df.ASMI.notna(),(high&low).astype(float),np.nan)
numeric=['EDAD','CINTURA21_1','CIRCPANTORRILLA19_1','CRP','FERRITINA','HCST','VIT_B12','VIT_D']; categorical=['SEXO','ENT','REGION']
exclusions={'SO_SIMP_1':{'BMI','CIRCPANTORRILLA19_1','SEXO','WEIGHT_FINAL','HEIGHT_FINAL','BFP','ASMI'},'SO_ESPEN':{'BMI','BFP','ASMI','WEIGHT_FINAL','HEIGHT_FINAL','EDAD','SEXO','CIRCPANTORRILLA19_1','CINTURA21_1'}}
biochemical=['CRP','FERRITINA','HCST','VIT_B12','VIT_D','ENT','REGION']
def pipe(ns,cs,iterative=False):
    imp=IterativeImputer(random_state=42) if iterative else SimpleImputer(strategy='median')
    return Pipeline([('prep',ColumnTransformer([('num',Pipeline([('imp',imp),('scale',StandardScaler())]),ns),('cat',Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('ohe',OneHotEncoder(handle_unknown='ignore',drop='first'))]),cs)])),('model',LogisticRegression(class_weight='balanced',max_iter=2000,random_state=42))])
cv=StratifiedKFold(5,shuffle=True,random_state=42); metrics=[]
for target in ['SO_SIMP_1','SO_ESPEN']:
    d=df.dropna(subset=[target]); y=d[target].astype(int); prev=y.mean(); ns=[c for c in numeric if c not in exclusions[target] and c in d]; cs=[c for c in categorical if c not in exclusions[target] and c in d]
    for label, cols in [('full',ns+cs),('biochemical',[c for c in biochemical if c in d])]:
        n=[c for c in cols if c in ns]; cat=[c for c in cols if c in cs]; scores=cross_validate(pipe(n,cat),d[cols],y,cv=cv,scoring={'roc':'roc_auc','pr':'average_precision'},n_jobs=-1)
        metrics.append({'target':target,'model':label,'N':len(d),'prevalence':prev,'PR_AUC_baseline':prev,'ROC_AUC_mean':scores['test_roc'].mean(),'ROC_AUC_sd':scores['test_roc'].std(),'PR_AUC_mean':scores['test_pr'].mean(),'PR_AUC_sd':scores['test_pr'].std()})
pd.DataFrame(metrics).to_csv(RESULTS_DIR/'cv_metrics_with_pr_baseline.csv',index=False); print(pd.DataFrame(metrics))


In [ ]:
clinical_num=['EDAD','CINTURA21_1','CRP','FERRITINA','HCST','VIT_B12','VIT_D']; clinical_cat=['SEXO','ENT','REGION']
def fit_or(target,data,cols,label):
    d=data.dropna(subset=[target]+cols); y=d[target].astype(float); X=pd.get_dummies(d[cols],columns=[c for c in clinical_cat if c in cols],drop_first=True,dtype=float); X=sm.add_constant(X,has_constant='add'); fit=sm.Logit(y,X).fit(disp=False,maxiter=200); return pd.DataFrame({'Feature':fit.params.index,'OR':np.exp(fit.params.values),'P_value':fit.pvalues.values,'N':len(d),'Analysis':label,'Condition_Number':np.linalg.cond(X.to_numpy())})
ors=[]
for target in ['SO_SIMP_1','SO_ESPEN']:
    try: ors.append(fit_or(target,df,clinical_num+clinical_cat,'complete_case'))
    except (np.linalg.LinAlgError,ValueError) as e: print(target,'complete-case failed:',e)
    d=df.dropna(subset=[target]).copy(); xi=pd.DataFrame(IterativeImputer(random_state=42).fit_transform(d[clinical_num]),index=d.index,columns=clinical_num); cats=d[clinical_cat].fillna({c:d[c].mode().iloc[0] for c in clinical_cat}); di=pd.concat([xi,cats],axis=1); di[target]=d[target]
    try: ors.append(fit_or(target,di,clinical_num+clinical_cat,'iterative_imputation_sensitivity'))
    except (np.linalg.LinAlgError,ValueError) as e: print(target,'imputed failed:',e)
pd.concat(ors,ignore_index=True).to_csv(RESULTS_DIR/'clinical_ORs_complete_case_and_sensitivity.csv',index=False)


In [ ]:
weight=next((c for c in ['FACTOR','FACTOR_EXPANSION'] if c in df and df[c].gt(0).any()),None)
if weight:
    out=[]
    for target in ['SO_SIMP_1','SO_ESPEN']:
        d=df.dropna(subset=[target,weight]); out.append({'target':target,'N':len(d),'weight':weight,'unweighted_prevalence':d[target].mean(),'weighted_prevalence':np.average(d[target],weights=d[weight])})
    pd.DataFrame(out).to_csv(RESULTS_DIR/'weighted_prevalence.csv',index=False); print(pd.DataFrame(out))
else: print('No positive FACTOR/FACTOR_EXPANSION column; weighted estimates unavailable.')
